# Apache Spark Structured Streaming

> **适用场景**: 实时数据流处理、流批一体
> **面试频率**: ⭐⭐⭐⭐⭐ 极高频

## 目录
1. Micro-batch vs Continuous Processing
2. Watermark & 迟到数据处理
3. Trigger 类型
4. Checkpointing & Exactly-once
5. Stateful Aggregation & State Store
6. 练习题

---
## 1. Micro-batch vs Continuous Processing

### Micro-batch（默认模式）
- 将流数据按**固定时间间隔**划分为小批次处理
- 延迟：~100ms 到数秒
- 支持 **Exactly-once** 语义
- 支持所有 SQL 操作和聚合

```
时间轴: ─────────────────────────────▶
数据流: ●●●|●●●●|●|●●●●●|●●
批次:       [t0] [t1]  [t2] [t3]
```

### Continuous Processing（Spark 2.3+，实验性）
- 记录级别处理，延迟低至 **~1ms**
- 仅支持 **At-least-once**（通过异步 epoch checkpoint）
- 功能限制：不支持聚合、不支持部分 Source/Sink
- 生产使用较少

```python
# Continuous Processing 触发方式
query = df.writeStream \
    .trigger(continuous='1 second') \
    .start()
```

### 选择建议
- 延迟要求 > 100ms：用 **Micro-batch**（稳定、功能完整）
- 极低延迟且简单 ETL：考虑 Continuous Processing（实验性）

---
## 2. Watermark & 迟到数据处理

### 问题：迟到数据
流式系统中，事件的**发生时间（event time）**和**到达时间（processing time）**可能不同。  
如果一条 10:00 发生的事件在 10:05 才到达，按事件时间聚合时会遇到问题。

### Watermark 机制
Watermark = `当前最大事件时间 - 允许迟到时间`

只有事件时间 ≥ Watermark 的数据才会被处理；更早的数据被**丢弃**（或触发 late event 处理）。

```python
from pyspark.sql.functions import window, col

query = spark.readStream \
    .format('kafka') \
    .option('subscribe', 'events') \
    .load() \
    .selectExpr('CAST(value AS STRING)', 'timestamp') \
    .withWatermark('timestamp', '10 minutes') \  # 允许最多迟到 10 分钟
    .groupBy(
        window(col('timestamp'), '5 minutes'),   # 5 分钟滚动窗口
        col('user_id')
    ) \
    .count() \
    .writeStream \
    .outputMode('append') \
    .format('parquet') \
    .option('path', '/output/') \
    .option('checkpointLocation', '/checkpoint/') \
    .start()
```

### Watermark 与 Output Mode
| Output Mode | 含义 | 是否需要 Watermark |
|-------------|------|------------------|
| **append** | 只输出已完成窗口的最终结果 | **必须** |
| **update** | 每次触发输出更新的行 | 可选（建议加，否则 state 无限增长） |
| **complete** | 每次触发输出全量聚合结果 | 不支持 Watermark（state 永不清理） |

---
## 3. Trigger 类型

```python
from pyspark.sql.streaming import Trigger

# 1. 默认（尽快处理，无间隔）
query = df.writeStream.trigger(processingTime='0 seconds').start()

# 2. 固定间隔 Micro-batch
query = df.writeStream.trigger(processingTime='1 minute').start()
# 每分钟处理一个 batch，即使数据量少也等满 1 分钟

# 3. Once（处理一次就停止）
query = df.writeStream.trigger(once=True).start()
query.awaitTermination()
# 适合：定时批量补跑，像 batch 一样触发一次

# 4. AvailableNow（Spark 3.3+，改进版 Once）
query = df.writeStream.trigger(availableNow=True).start()
# 处理当前所有可用数据后停止（多个 micro-batch，更均匀）

# 5. Continuous Processing
query = df.writeStream.trigger(continuous='1 second').start()
# 每 1 秒 checkpoint 一次（非批次间隔）
```

### 选择建议
| 场景 | 推荐 Trigger |
|------|-------------|
| 实时 Dashboard | `processingTime='5 seconds'` |
| 定时 ETL（每小时）| `availableNow=True` + 调度器 |
| 高吞吐、延迟不敏感 | `processingTime='5 minutes'`（合并更多数据）|
| 极低延迟 | 默认（`0 seconds`）|

---
## 4. Checkpointing & Exactly-once

### Checkpoint 存储什么？
```
checkpointLocation/
├── commits/          ← 已完成的 batch ID
├── offsets/          ← 每个 batch 的 Source 偏移量（Kafka offset 等）
├── metadata          ← query 元数据
└── state/            ← Stateful 聚合的状态数据
```

### Exactly-once 实现条件
三个组件都需要保证：

1. **Source**：支持重放（replay）
   - ✅ Kafka（可重置 offset）
   - ✅ 文件系统（幂等读取）
   - ❌ Socket（无法重放）

2. **Spark 引擎**：Checkpoint 保证 offset 与 state 一致性

3. **Sink**：幂等写入或事务写入
   - ✅ 文件系统（overwrite 模式）
   - ✅ Delta Lake（ACID 事务）
   - ✅ Kafka（`enable.idempotence=true`）
   - ❌ 普通 JDBC（需自行保证幂等）

```python
query = df.writeStream \
    .format('delta') \
    .option('checkpointLocation', 'gs://bucket/checkpoint/job1') \
    .outputMode('append') \
    .start('gs://bucket/output/table1')
```

---
## 5. Stateful Aggregation & State Store

### 什么是有状态操作？
需要跨 batch 记录中间状态的操作：
- 时间窗口聚合（window aggregation）
- `dropDuplicates()`
- `flatMapGroupsWithState` / `mapGroupsWithState`
- Stream-Stream Join

### State Store
- 默认后端：**RocksDB**（Spark 3.2+ 推荐）或 HDFS
- 存储每个 key 的聚合中间值
- 定期 snapshot 到 checkpoint 目录

```python
# 启用 RocksDB State Store（更高效，支持大状态）
spark.conf.set(
    'spark.sql.streaming.stateStore.providerClass',
    'org.apache.spark.sql.execution.streaming.state.RocksDBStateStoreProvider'
)
```

### State 无限增长问题
```python
# ❌ 无 Watermark 的聚合：state 永远不会清理
df.groupBy('user_id').count()

# ✅ 加 Watermark：过期 state 自动清理
df.withWatermark('event_time', '1 hour') \
  .groupBy(window('event_time', '10 minutes'), 'user_id') \
  .count()

# ✅ 自定义状态超时（flatMapGroupsWithState）
# 可以设置 GroupState.setTimeoutDuration() 精细控制
```

### 监控 State
```python
# 查看 state 大小
query.lastProgress['stateOperators']
# 输出: [{'numRowsTotal': 1000000, 'numRowsUpdated': 5000, ...}]
```

---
## 6. 练习题

### Q1 [高频] Watermark 是什么？不设置 Watermark 会有什么问题？

<details><summary>参考答案</summary>

Watermark = 最大事件时间 - 允许迟到阈值。它告诉 Spark：早于 Watermark 的数据可以认为不会再来了，相关的窗口状态可以安全清理并输出结果。

**不设置的问题**：
1. 使用 `append` 模式的聚合会报错（append 模式需要知道窗口何时关闭）
2. State Store 持续增长，最终导致 OOM
3. 窗口永远不会输出最终结果（无法判断数据是否到齐）
</details>

---

### Q2 [高频] Exactly-once 需要哪三个条件？

<details><summary>参考答案</summary>

1. **Source 可重放**：出错后能从上次的 offset 重新读取（如 Kafka offset replay）
2. **引擎状态一致**：Checkpoint 保证 offset 和 state 的原子提交
3. **Sink 幂等/事务**：重复写入不产生重复数据（如 Delta Lake ACID、Kafka idempotent producer）

任一环节缺失，最多只能保证 At-least-once。
</details>

---

### Q3 Output Mode 的三种类型及适用场景？

<details><summary>参考答案</summary>

- **append**：只追加新的完整结果（窗口关闭后才输出）。需要 Watermark。适合：写入数据湖、不可变记录。
- **update**：每次 micro-batch 输出有变化的行。适合：实时 Dashboard、数据库 upsert。
- **complete**：每次输出全量聚合结果（覆盖写）。适合：全局排行榜，小数据量聚合。State 永不清理，不适合无界聚合。
</details>

---

### Q4 Streaming 作业重启后从哪里继续？如何保证不丢数据？

<details><summary>参考答案</summary>

Spark Structured Streaming 通过 **Checkpoint** 保存：
- 上次处理到的 Source offset（如 Kafka topic partition offset）
- 聚合 State 快照
- 已提交的 batch ID

重启后，Spark 读取 checkpoint 中的 offset，**从该 offset 继续消费**，State 也从快照恢复。

保证不丢数据的前提：
- Source 数据在 checkpoint offset 位置之后仍然可读（Kafka 消息未过期）
- Checkpoint 目录可靠存储（S3/HDFS，不能用本地磁盘）
</details>

---

### Q5 State Store 增长过快如何处理？

<details><summary>参考答案</summary>

1. **设置 Watermark**：自动清理过期 state（最重要）
2. **使用 RocksDB State Store**：比默认内存 state store 更省内存，支持更大 state
3. **自定义超时**：用 `flatMapGroupsWithState` + `GroupState.setTimeoutDuration()` 精细控制 key 的生命周期
4. **监控**：`query.lastProgress['stateOperators']` 观察 state row 数量趋势
5. **缩短窗口时长**：减少同时维护的窗口数量
</details>